# Structured Data with Gemini

This notebook covers how to get Gemini to output structured data (JSON) and how to handle Markdown responses. Structured output is essential for building reliable applications where you need to parse the model's response programmatically.

In [ ]:
import os
import json
import sys
from pathlib import Path
from dotenv import load_dotenv
from google import genai
from google.genai import types
from IPython.display import display, Markdown
from pydantic import BaseModel

project_root = next(
    (path for path in [Path.cwd(), *Path.cwd().parents] if (path / "requirements.txt").exists()),
    Path.cwd(),
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from utils.gemini_retry import generate_content_with_retry

# 1. Load API Key from .env
load_dotenv()
api_key = os.getenv("GOOGLE_API_KEY")

# 2. Initialize the Client
client = genai.Client(api_key=api_key)

## 1. Outputting JSON (JSON Mode)

The simplest way to get JSON is to set the `response_mime_type` to `application/json` in the `GenerateContentConfig`. This ensures the model returns a valid JSON string.

Gemini calls in this notebook now use the shared Tenacity-based retry helper for `429` and `RESOURCE_EXHAUSTED` responses.

In [ ]:
prompt = "List 3 famous landmarks in Paris with their construction year."

response = generate_content_with_retry(
    client=client,
    model="gemini-2.5-flash",
    contents=prompt,
    config=types.GenerateContentConfig(
        response_mime_type="application/json"
    ),
)

# The response.text is a JSON string
json_data = json.loads(response.text)
display(json_data)

[{'landmark': 'Eiffel Tower', 'year': '1889'},
 {'landmark': 'Notre-Dame Cathedral', 'year': '1345'},
 {'landmark': 'Arc de Triomphe', 'year': '1836'}]

## 2. Controlled JSON with Response Schema

For more precise control over the JSON structure, you can provide a `response_schema`. This forces the model to follow a specific shape, which is critical for downstream processing.

In [ ]:
class Landmark(BaseModel):
    name: str
    year: int
    description: str

prompt = "Give me details about the Eiffel Tower."

response = generate_content_with_retry(
    client=client,
    model="gemini-2.5-flash",
    contents=prompt,
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=Landmark
    ),
)

landmark_info = json.loads(response.text)
display(landmark_info)

{'name': 'Eiffel Tower',
 'year': 1889,
 'description': "The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris, France. It is named after the engineer Gustave Eiffel, whose company designed and built the tower. Constructed from 1887 to 1889 as the entrance arch to the 1889 World's Fair, it was initially criticized by some of France's leading artists and intellectuals for its design, but it has become a global cultural icon of France and one of the most recognizable structures in the world."}

## 3. Combining Markdown and JSON

Sometimes you want a human-readable explanation (Markdown) alongside raw data (JSON). You can prompt the model to provide both, often by using a schema that includes a field for the explanation.

In [ ]:
class HybridResponse(BaseModel):
    explanation_md: str
    data: list[Landmark]

prompt = "Analyze 2 landmarks in Rome. Provide a brief Markdown analysis and then the data in JSON."

response = generate_content_with_retry(
    client=client,
    model="gemini-2.5-flash",
    contents=prompt,
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=HybridResponse
    ),
)

result = json.loads(response.text)

display(Markdown("### Markdown Explanation"))
display(Markdown(result['explanation_md']))

display(Markdown("### Structured Data (JSON)"))
display(result['data'])

### Markdown Explanation

Rome, the Eternal City, is a treasure trove of ancient landmarks that tell the story of one of history's most powerful empires. Among its most iconic structures are the Colosseum and the Roman Forum. The Colosseum stands as a testament to Roman engineering and their appetite for grand spectacles, serving as the primary venue for gladiatorial contests and public entertainment. Its imposing structure still dominates the cityscape. The Roman Forum, on the other hand, represents the heart of ancient Rome's political, religious, and commercial life. Though largely in ruins today, walking through its remnants allows one to visualize the bustling center of an empire, where crucial decisions were made and daily life unfolded for centuries. Together, these two sites offer invaluable insights into the daily life, governance, and entertainment of ancient Roman society.

### Structured Data (JSON)

[{'name': 'Colosseum',
  'year': 80,
  'description': 'An elliptical amphitheatre in the centre of the city of Rome, Italy. Built of travertine limestone, tuff, and brick-faced concrete, it was the largest amphitheatre ever built and held 50,000 to 80,000 spectators. It was primarily used for gladiatorial contests and public spectacles.'},
 {'name': 'Roman Forum',
  'year': -500,
  'description': 'A rectangular forum (plaza) surrounded by the ruins of several important ancient government buildings at the center of the city of Rome. For centuries, it was the center of Roman public life, including triumphal processions, elections, public speeches, and commercial affairs.'}]

## Best Practices for Structured Data

- **Use Schema:** Always prefer `response_schema` over just `response_mime_type` for reliability.
- **Clear Descriptions:** If using complex schemas, include descriptions in the schema definition to guide the model.
- **Type Consistency:** Ensure your prompt aligns with the expected types (e.g., don't ask for a list if the schema expects a single object).
- **Model Choice:** `gemini-2.5-flash` is excellent and fast for structured output tasks.